<a href="https://colab.research.google.com/github/dgaida/wpf_cps_th_public/blob/main/assets/exercises/week_02/01_monokulare_tiefenschaetzung_depth_pro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Woche 2: Monokulare Tiefenschätzung mit Depth Pro (Apple Research)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/wpf_cps_th_public/blob/main/assets/exercises/week_02/01_monokulare_tiefenschaetzung_depth_pro.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/dgaida/wpf_cps_th_public/blob/main/assets/exercises/week_02/01_monokulare_tiefenschaetzung_depth_pro.ipynb)

In diesem Notebook untersuchen wir den aktuellen Stand der Technik (**State of the Art, SOTA**) in der **monokularen Tiefenschätzung** (Monocular Depth Estimation, MDE). Wir verwenden das von Apple AI/ML entwickelte Open-Source-Modell **Depth Pro** (*Bochkovskii et al., 2024*), welches hochauflösende, metrische Tiefenkarten mit extrem scharfen Objektkanten in unter einer Sekunde erzeugt.

## 1. Theoretischer Hintergrund & State of the Art (2024/2025)

### 1.1 Was ist monokulare Tiefenschätzung?
Die monokulare Tiefenschätzung ist die Aufgabe, für jeden Pixel $i=(x, y)$ eines einzelnen 2D-RGB-Bildes $I$ die reale Entfernung $Z(x, y)$ zur Kamera zu schätzen. Da bei der Projektion einer 3D-Welt auf eine 2D-Bildebene die Tiefeninformation verloren geht, handelt es sich um ein mathematisch unterbestimmtes (*ill-posed*) Problem. Moderne Deep-Learning-Modelle lösen dieses Problem, indem sie visuelle Hinweise wie Perspektive, Texturverläufe, Objektgrößen und Schatten interpretieren.

### 1.2 Metrische vs. Relative Tiefenschätzung
- **Relative Tiefenschätzung (Disparität / Skaleninvarianz):** Gibt an, welche Objekte im Bild näher oder weiter entfernt sind (z. B. Wertebereich $[0, 1]$), jedoch ohne reale physikalische Maßeinheiten.
- **Metrische Tiefenschätzung:** Sagt die tatsächliche absolute Entfernung in **Metern ($m$)** voraus.

### 1.3 Vergleich moderner SOTA-Modelle
| Modell | Jahr / Konferenz | Typ | Auflösung & Kanten | Brennweite ($f_{px}$) | Geschwindigkeit |
| :--- | :---: | :---: | :---: | :---: | :---: |
| [**Apple Depth Pro**](https://github.com/apple-aiml-research/ml-depth-pro) | Oct 2024 (ICLR 2025) | Metrisch (Zero-Shot) | Hochauflösend (2.25 MP), extrem scharf | Automatisch geschätzt | **Sehr schnell (<0,3 s auf GPU)** |
| [**Depth Anything V2**](https://github.com/DepthAnything/Depth-Anything-V2) | Juni 2024 (NeurIPS 2024) | Relativ / Metrisch | Hohe Genauigkeit (DINOv2) | Externe Parameter erforderlich | Schnell |
| [**Marigold**](https://github.com/prs-eth/marigold) | CVPR 2024 | Latent Diffusion | Feine Details & Texturen | Nicht direkt metrisch | Langsam (iterative Diffusion) |
| [**Metric3D v2**](https://jugghm.github.io/Metric3Dv2/) | 2024 | Metrisch & Normalen | Hohe 3D-Geometrie-Treue | Über gegebene Kamera-Intrinsics | Mittel |

### 1.4 Highlights von Depth Pro (Apple Research)
1. **Keine Kamerametadaten nötig:** Geschätzte metrische Tiefe in Metern ohne Notwendigkeit von EXIF-Daten oder bekannten Kameraparametern.
2. **Integrierte Brennweitenschätzung:** Das Modell schätzt die Brennweite $f_{px}$ direkt aus dem RGB-Bild.
3. **Hohe Kantenpräzision:** Durch ein Multi-Scale Vision Transformer Framework bleiben scharfe Kanten an Objektgrenzen erhalten.

## 2. Setup und Installation

Wir klonen das `ml-depth-pro`-Repository von Apple und installieren das Modell.

### 💡 Warum wird `numpy<2` aus `pyproject.toml` entfernt?
1. **Kein Downgrade von PyTorch / CUDA erforderlich:** Google Colab stellt standardmäßig PyTorch 2.x und CUDA 12 vorkonfiguriert bereit. Ein Herabstufen von PyTorch oder NumPy auf 1.x würde das Herunterladen von >2 GB alten Wheels erzwingen und ein Neustarten der Python-Laufzeit erforderlich machen.
2. **Kompatibilität mit Colab-Paketen:** Vorinstallierte Bilbliotheken wie OpenCV (`opencv-python 5.0`) und JAX in Colab setzen NumPy 2.x voraus. Ein Downgrade von NumPy erzeugt Konflikte mit diesen Paketen.
3. **Entfernung der veralteten Pinning-Hürde:** Die `numpy<2` Beschränkung in Depth Pro entstand beim Release im Sommer 2024 als vorsorgliche Pinning-Regel. Der Python-Code von Depth Pro selbst läuft absolut fehlerfrei auf NumPy 2.x und modernem PyTorch!

In [ ]:
import os
import sys
import subprocess
import torch

# GPU-Verfügbarkeit prüfen
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA verfügbar: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Modell: {torch.cuda.get_device_name(0)}")

# Depth Pro Repository klonen und pyproject.toml anpassen
if not os.path.exists("ml-depth-pro"):
    !git clone -q https://github.com/apple-aiml-research/ml-depth-pro.git
    !sed -i 's/"numpy<2",//g' ml-depth-pro/pyproject.toml

# Paket direkt und regulär installieren (ohne -e), um Importfehler in Colab zu vermeiden
!pip install -q ./ml-depth-pro

# Verzeichnis für Gewichte erstellen
os.makedirs("checkpoints", exist_ok=True)
checkpoint_path = "checkpoints/depth_pro.pt"

# Falls die Datei existiert, aber leer oder fehlerhaft ist, löschen wir sie
if os.path.exists(checkpoint_path) and os.path.getsize(checkpoint_path) < 1024 * 1024:
    print("Beschädigte oder leere Modelldatei (0 Bytes) erkannt. Lösche und starte Download von offizieller Quelle neu...")
    os.remove(checkpoint_path)

if not os.path.exists(checkpoint_path):
    print("Lade vortrainiertes Depth Pro Modell von der offiziellen Apple-Quelle herunter...")
    # Verwende die offizielle Apple CDN URL
    url = "https://ml-site.cdn-apple.com/models/depth-pro/depth_pro.pt"
    !wget -q -O {checkpoint_path} {url}
    print("Download abgeschlossen!")
else:
    print("Modellgewicht `depth_pro.pt` ist bereits vorhanden und intakt.")

## 3. Laden des Depth Pro Modells

Wir laden das vortrainierte Modell sowie die Preprocessing-Transformationen mithilfe der von Apple bereitgestellten Standardkonfiguration (`DEFAULT_MONODEPTH_CONFIG_DICT`).

In [ ]:
import os
import sys
import torch
import depth_pro

# Device festlegen (GPU wenn verfügbar)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Verwende Device: {device}")

# Modell und Bildtransformationen mit offiziellen Standard-Gewichten laden
model, transform = depth_pro.create_model_and_transforms(device=device)
model.eval()

print("Depth Pro Modell erfolgreich geladen!")

## 4. Bildeingabe: Beispielbild oder eigenes Bild hochladen

Sie können entweder ein mitgeliefertes Beispielbild verwenden oder ein eigenes Bild hochladen.

In [ ]:
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt

# Verzeichnis für Eingabebilder erstellen
os.makedirs("data", exist_ok=True)
sample_image_path = "data/example_street.jpg"

# Beispielbild herunterladen, falls nicht vorhanden
if not os.path.exists(sample_image_path):
    sample_url = "https://raw.githubusercontent.com/apple-aiml-research/ml-depth-pro/main/data/example.jpg"
    response = requests.get(sample_url)
    img = Image.open(BytesIO(response.content)).convert("RGB")
    img.save(sample_image_path)
    print(f"Beispielbild gespeichert unter: {sample_image_path}")

# Optionale Interaktion für Google Colab
use_custom_image = False  # Setzen Sie diesen Wert auf True, um in Colab ein eigenes Bild hochzuladen

if use_custom_image:
    try:
        from google.colab import files
        uploaded = files.upload()
        for filename in uploaded.keys():
            image_path = filename
            print(f"Eigenes Bild hochgeladen: {image_path}")
            break
    except ImportError:
        print("Google Colab Umgebung nicht erkannt. Verwende Beispielbild.")
        image_path = sample_image_path
else:
    image_path = sample_image_path

print(f"Verwende Bild für Inferenz: {image_path}")

# Bild anzeigen
rgb_image = Image.open(image_path).convert("RGB")
plt.figure(figsize=(8, 6))
plt.imshow(rgb_image)
plt.title("Eingabebild (RGB)")
plt.axis("off")
plt.show()

## 5. Inferenz & Schätzung der metrischen Tiefe

Wir führen die Inferenz mit Depth Pro aus. Das Modell liefert:
1. Eine **metrische Tiefenkarte ($Z$)** mit Pixelwerten in Metern ($m$).
2. Die geschätzte **Brennweite ($f_{px}$)** der Kamera in Pixeln.

In [ ]:
import time

# Bild und geschätzte f_px laden
image, _, f_px = depth_pro.load_rgb(image_path)
image = transform(image)

# Inferenz ausführen und Zeit messen
start_time = time.time()
with torch.no_grad():
    prediction = model.infer(image, f_px=f_px)
elapsed_time = time.time() - start_time

# Ergebnisse extrahieren
depth_tensor = prediction["depth"]  # Tensor der Tiefe in [m]
depth_map = depth_tensor.squeeze().cpu().numpy()  # Numpy Array
focallength_px = prediction["focallength_px"].item() if isinstance(prediction["focallength_px"], torch.Tensor) else prediction["focallength_px"]

print(f"Inferenzzeit: {elapsed_time:.3f} Sekunden")
print(f"Geschätzte Brennweite (f_px): {focallength_px:.1f} Pixel")
print(f"Tiefenbereich: Min = {depth_map.min():.2f} m, Max = {depth_map.max():.2f} m, Mittelwert = {depth_map.mean():.2f} m")

## 6. Visualisierung der Ergebnisse

Wir stellen das Eingabebild und die geschätzte Tiefenkarte nebeneinander dar. Eine Farbskala gibt die Entfernung in Metern ($m$) an.

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Original RGB-Bild
axes[0].imshow(rgb_image)
axes[0].set_title("Eingabebild (RGB)", fontsize=14)
axes[0].axis("off")

# Metrische Tiefenkarte
im = axes[1].imshow(depth_map, cmap="plasma")
axes[1].set_title("Geschätzte metrische Tiefe (Depth Pro)", fontsize=14)
axes[1].axis("off")

# Colorbar mit Meter-Angabe
cbar = fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label("Tiefe in Metern (m)", fontsize=12)

plt.tight_layout()
plt.show()

## 7. Optionale 3D-Punktwolken-Visualisierung

Mithilfe der geschätzten Tiefenkarte $Z(x,y)$ und der geschätzten Brennweite $f_{px}$ können wir das 2D-Bild in eine **3D-Punktwolke $(X, Y, Z)$** zurückprojizieren:

$$X = \frac{(x - c_x) \cdot Z}{f_{px}}, \quad Y = \frac{(y - c_y) \cdot Z}{f_{px}}$$

In [ ]:
# 3D-Punktwolke generieren (subsampled für flüssige Darstellung)
step = 8  # Schrittweite zur Reduktion der Punktanzahl
h, w = depth_map.shape
c_x, c_y = w / 2.0, h / 2.0

# Gitter erstellen
y_coords, x_coords = np.mgrid[0:h:step, 0:w:step]
z_vals = depth_map[::step, ::step]
rgb_vals = np.array(rgb_image.resize((w, h)))[::step, ::step] / 255.0

# 3D-Koordinaten berechnen
x_vals = (x_coords - c_x) * z_vals / focallength_px
y_vals = (y_coords - c_y) * z_vals / focallength_px

# 3D-Plot mit Matplotlib
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

# Flachklopfen der Arrays
x_flat = x_vals.flatten()
y_flat = y_vals.flatten()
z_flat = z_vals.flatten()
colors = rgb_vals.reshape(-1, 3)

# Filter gegen Ausreißer
valid = (z_flat > 0) & (z_flat < np.percentile(z_flat, 98))

sc = ax.scatter(
    x_flat[valid],
    z_flat[valid],  # Z als Tiefe nach vorne
    -y_flat[valid], # -Y für korrekte Ausrichtung oben/unten
    c=colors[valid],
    s=1,
    alpha=0.8
)

ax.set_xlabel("X (m)")
ax.set_ylabel("Z / Tiefe (m)")
ax.set_zlabel("Y (m)")
ax.set_title("Rekonstruierte 3D-Punktwolke")
ax.view_init(elev=20, azim=-60)
plt.show()

### 7.1 Interaktive 3D-Visualisierung mit Plotly

Mit dem folgenden interaktiven Plotly-Plot können Sie die Punktwolke mit gedrückter linker Maustaste frei drehen, mit der rechten Maustaste verschieben und mit dem Mausrad zoomen.

In [ ]:
import plotly.graph_objects as go

# Nur die gültigen Punkte filtern
x_interactive = x_flat[valid]
y_interactive = y_flat[valid]
z_interactive = z_flat[valid]
colors_interactive = colors[valid]

# RGB-Werte in das von Plotly erwartete Format konvertieren (0-255)
colors_plotly = [f"rgb({int(r*255)}, {int(g*255)}, {int(b*255)})" for r, g, b in colors_interactive]

# 3D Scatter Plot erstellen
fig_interactive = go.Figure(
    data=[go.Scatter3d(
        x=x_interactive,
        y=z_interactive,      # Z als Tiefe nach vorne
        z=-y_interactive,     # -Y für korrekte Ausrichtung oben/unten
        mode='markers',
        marker=dict(
            size=1.5,
            color=colors_plotly,
            opacity=0.8
        )
    )]
)

# Layout konfigurieren für ein besseres visuelles Erlebnis
fig_interactive.update_layout(
    title="Interaktive 3D-Punktwolke (Drehbar mit Maus)",
    scene=dict(
        xaxis_title="X (m)",
        yaxis_title="Z / Tiefe (m)",
        zaxis_title="Y (m)",
        aspectmode='data' # Hält die Achsen im korrekten physikalischen Verhältnis zueinander
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    width=900,
    height=700
)

# Interaktive Grafik anzeigen
fig_interactive.show()

## 8. Zusammenfassung

- **Depth Pro** ist ein State-of-the-Art Foundation Modell für monokulare Tiefenschätzung von Apple AI/ML.
- Es berechnet **metrische Tiefenwerte in Metern ($m$)** und schätzt automatisch die **Brennweite $f_{px}$** der Kamera.
- Durch die hohe Auflösung (2.25 MP) und kantenbasierte Vorhersage eignet es sich hervorragend für mobile Robotik, AR/VR und 3D-Rekonstruktion.